# 🎨 Indie Comic Pipeline — Full Auto Run

**One-click execution. No manual steps required.**

This notebook installs all dependencies, sets up Ollama + Llama 3.2, and runs the complete Indie Comic pipeline:

| Step | Module | Description |
|------|--------|-------------|
| 1️⃣ | `character_extractor.py` | Extract personality traits from any fictional character |
| 2️⃣ | `story_extractor.py` | Extract world details from any story / universe |
| 3️⃣ | `fusion_engine.py` | Fuse character + setting into a 10-page storyboard |
| 4️⃣ | `emotion_recognition_engine.py` | Annotate every panel with per-character emotions |
| 5️⃣ | `generate_panels.py` | Generate comic panels with SDXL (GPU required) |
| 6️⃣ | Visualise | Display the generated comic panel grid inline |

---
⚠️ **Runtime:** Make sure you are connected to a **GPU runtime** (`Runtime → Change runtime type → T4 GPU`) before running this notebook.

---

## ⚙️ Cell 1 — Configuration

**Edit the values below to customise your comic. Then run all cells.**

In [ ]:
# ============================================================
#  🎭  PIPELINE CONFIGURATION  —  Edit these values
# ============================================================

CHARACTER_NAME = "Spider-Man"        # Any fictional character
STORY_WORLD    = "Cyberpunk 2077"    # Any story / universe / setting

# Which storyboard page to render panels for (1–10)
PAGE_TO_RENDER = 1

# SDXL image dimensions (must be multiples of 8, keep ≤1024 on T4)
IMG_WIDTH  = 768
IMG_HEIGHT = 768

# Number of diffusion steps (lower = faster, higher = better quality)
INFERENCE_STEPS = 30

# Guidance scale (how strongly the prompt is followed)
GUIDANCE_SCALE = 7.5

# Reproducibility seed
SEED = 42

# Ollama model (llama3.2 is small & fast; swap for llama3.1 for richer outputs)
OLLAMA_MODEL = "llama3.2"

print(f"✅ Config loaded: {CHARACTER_NAME} × {STORY_WORLD} | Page {PAGE_TO_RENDER}")
print(f"   Image: {IMG_WIDTH}×{IMG_HEIGHT}px | Steps: {INFERENCE_STEPS} | Seed: {SEED}")

## 📦 Cell 2 — Install Dependencies

Installs all Python packages needed by the pipeline. This takes **~3–5 minutes** on first run.

In [ ]:
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

print("📦 Installing core ML libraries...")
pip_install(
    "diffusers==0.29.2",
    "accelerate",
    "transformers",
    "safetensors",
    "xformers",
    "peft",
    "torch",
    "torchvision",
)

print("📦 Installing LangChain + Ollama...")
pip_install(
    "langchain",
    "langchain-core",
    "langchain-ollama",
    "langchain-community",
    "ollama",
)

print("📦 Installing image processing + utilities...")
pip_install(
    "Pillow",
    "opencv-python-headless",
    "numpy",
    "scipy",
    "scikit-image",
    "PyYAML",
    "requests",
)

print("✅ All Python packages installed!")

## 🦙 Cell 3 — Install & Start Ollama + Pull Llama 3.2

In [ ]:
import subprocess, os, time, socket, threading

# ── 1. Download and install Ollama binary ─────────────────────────────────────
print("🦙 Installing Ollama...")
subprocess.run(
    "curl -fsSL https://ollama.com/install.sh | sh",
    shell=True, check=True
)
print("✅ Ollama binary installed.")

# ── 2. Launch the Ollama server in background ──────────────────────────────────
def start_ollama_server():
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

thread = threading.Thread(target=start_ollama_server, daemon=True)
thread.start()

# ── 3. Wait until the server is reachable ─────────────────────────────────────
def ollama_ready(host="localhost", port=11434, timeout=60):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            s = socket.create_connection((host, port), timeout=1)
            s.close()
            return True
        except OSError:
            time.sleep(1)
    return False

print("⏳ Waiting for Ollama server to be ready...")
if ollama_ready():
    print("✅ Ollama server is running on port 11434.")
else:
    raise RuntimeError("❌ Ollama server failed to start within 60 seconds.")

# ── 4. Pull the LLM model ──────────────────────────────────────────────────────
print(f"🦙 Pulling model '{OLLAMA_MODEL}' (this may take a few minutes)...")
result = subprocess.run(["ollama", "pull", OLLAMA_MODEL], capture_output=True, text=True)
if result.returncode != 0:
    print(f"⚠️ Pull stderr: {result.stderr}")
    raise RuntimeError(f"Failed to pull {OLLAMA_MODEL}")
print(f"✅ Model '{OLLAMA_MODEL}' is ready.")

## 📁 Cell 4 — Clone the Pipeline Repository

In [ ]:
import os, subprocess

REPO_URL   = "https://github.com/Cyberpunk-San/Indie-Comic.git"
REPO_DIR   = "/content/indie_comic_pipeline"
SUBDIR     = "indie_comic_pipeline"   # The actual pipeline sub-folder inside the repo

if not os.path.exists(REPO_DIR):
    print(f"📥 Cloning repo from {REPO_URL}...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    print("✅ Repository cloned.")
else:
    print("✅ Repository already present — pulling latest changes...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

# The pipeline lives inside a sub-directory
PIPELINE_ROOT = os.path.join(REPO_DIR, SUBDIR)
os.chdir(PIPELINE_ROOT)
print(f"📂 Working directory set to: {os.getcwd()}")

# Ensure output directories exist
for d in ["outputs/fusion", "outputs/comics", "outputs/characters"]:
    os.makedirs(d, exist_ok=True)
    
print("✅ Directory structure ready.")

## 🔧 Cell 5 — Patch settings.yaml with our config values

In [ ]:
import yaml, os

settings_path = os.path.join(os.getcwd(), "config", "settings.yaml")

with open(settings_path, "r") as f:
    settings = yaml.safe_load(f)

# Override generation settings from our config cell
settings.setdefault("generation", {})
settings["generation"]["default_size"] = {"width": IMG_WIDTH, "height": IMG_HEIGHT}
settings["generation"]["inference_steps"] = INFERENCE_STEPS
settings["generation"]["guidance_scale"] = GUIDANCE_SCALE
settings["generation"]["seed"] = SEED

# Override LangChain model
settings.setdefault("langchain", {})
settings["langchain"]["model"] = OLLAMA_MODEL
settings["langchain"]["ollama_url"] = "http://localhost:11434"

# CUDA is available in Colab — force GPU
settings.setdefault("models", {}).setdefault("sdxl", {})
settings["models"]["sdxl"]["device"] = "cuda"
settings["models"]["sdxl"]["memory_optimization"] = True

with open(settings_path, "w") as f:
    yaml.dump(settings, f, default_flow_style=False)

print("✅ settings.yaml patched with Colab runtime values.")
print(f"   SDXL: {IMG_WIDTH}×{IMG_HEIGHT} | {INFERENCE_STEPS} steps | device=cuda")
print(f"   LLM : {OLLAMA_MODEL} @ http://localhost:11434")

## 🎭 Cell 6 — Step 1: Character Personality Extraction

In [ ]:
import subprocess, sys, os

print("=" * 70)
print(f"STEP 1/5: Extracting personality for '{CHARACTER_NAME}'")
print("=" * 70)

result = subprocess.run(
    [sys.executable, "character_extractor.py", CHARACTER_NAME],
    cwd=os.path.join(os.getcwd(), "langchain_code"),
    capture_output=False,          # Stream output live
    text=True
)

if result.returncode != 0:
    raise RuntimeError("❌ Character extraction failed! Check the output above.")

# Show what was saved
import json
char_file = os.path.join(os.getcwd(), "outputs", "fusion", "character_personality.json")
with open(char_file) as f:
    char_data = json.load(f)

print("\n📋 Character Summary:")
print(f"  Name   : {char_data.get('character_name')}")
print(f"  Traits : {', '.join(char_data.get('core_personality_traits', []))}")
print(f"  Colors : {', '.join(char_data.get('signature_colors', []))}")
print("\n✅ Step 1 complete.")

## 🌍 Cell 7 — Step 2: Story Setting Extraction

In [ ]:
import subprocess, sys, os, json

print("=" * 70)
print(f"STEP 2/5: Extracting setting for '{STORY_WORLD}'")
print("=" * 70)

result = subprocess.run(
    [sys.executable, "story_extractor.py", STORY_WORLD],
    cwd=os.path.join(os.getcwd(), "langchain_code"),
    capture_output=False,
    text=True
)

if result.returncode != 0:
    raise RuntimeError("❌ Story extraction failed! Check the output above.")

story_file = os.path.join(os.getcwd(), "outputs", "fusion", "story_setting.json")
with open(story_file) as f:
    story_data = json.load(f)

print("\n📋 Setting Summary:")
print(f"  World   : {story_data.get('story_name')}")
print(f"  Era     : {story_data.get('era')}")
print(f"  Location: {story_data.get('location')}")
print(f"  Mood    : {story_data.get('mood')}")
print(f"  Colors  : {', '.join(story_data.get('color_palette', []))}")
print("\n✅ Step 2 complete.")

## ⚗️ Cell 8 — Step 3: Fusion Engine (10-Page Storyboard Generation)

In [ ]:
import subprocess, sys, os, json

print("=" * 70)
print(f"STEP 3/5: Fusing '{CHARACTER_NAME}' into '{STORY_WORLD}'")
print("         Generating 10-page storyboard... (takes 2-4 minutes)")
print("=" * 70)

result = subprocess.run(
    [sys.executable, "fusion_engine.py"],
    cwd=os.path.join(os.getcwd(), "langchain_code"),
    capture_output=False,
    text=True
)

if result.returncode != 0:
    raise RuntimeError("❌ Fusion engine failed! Check the output above.")

fusion_file = os.path.join(os.getcwd(), "outputs", "fusion", "fusion_complete.json")
with open(fusion_file) as f:
    fusion_data = json.load(f)

fusion = fusion_data.get("fusion", {})
pages  = fusion.get("storyboard_10_pages", [])
comps  = fusion.get("components", [])

print("\n📋 Fusion Summary:")
print(f"  Visual Look  : {fusion.get('character_visual_looks', '')[:120]}...")
print(f"  Pages Created: {len(pages)}")
print(f"  Key Assets   : {len(comps)}")
for c in comps:
    print(f"    [{c['type'].upper()}] {c['name']}")

print("\n✅ Step 3 complete.")

## 😮 Cell 9 — Step 4: Emotion Recognition & Panel Prompt Augmentation

In [ ]:
import subprocess, sys, os, json

print("=" * 70)
print("STEP 4/5: Annotating panels with emotions")
print("         Processing 10 pages × N panels each...")
print("=" * 70)

result = subprocess.run(
    [sys.executable, "emotion_recognition_engine.py"],
    cwd=os.path.join(os.getcwd(), "langchain_code"),
    capture_output=False,
    text=True
)

if result.returncode != 0:
    raise RuntimeError("❌ Emotion recognition failed! Check the output above.")

emotions_file = os.path.join(os.getcwd(), "outputs", "fusion", "storyboard_with_emotions.json")
with open(emotions_file) as f:
    em_data = json.load(f)

pages_annotated = em_data.get("storyboard_with_emotions", [])

print("\n📋 Emotion Summary (target page preview):")
target = next((p for p in pages_annotated if p.get("page_number") == PAGE_TO_RENDER), None)
if target:
    print(f"  Page {PAGE_TO_RENDER}: {target.get('location')}")
    for pd in target.get("panels_detail", [])[:3]:
        emos = pd.get("emotions", {})
        for char, emo in emos.items():
            if isinstance(emo, dict):
                print(f"  Panel {pd['panel_number']} | {char}: {emo.get('emotion')} ({emo.get('intensity')})")
                print(f"    Expr: {emo.get('expression_trigger', '')[:80]}")
else:
    print(f"  (Page {PAGE_TO_RENDER} not found in annotated data)")

print("\n✅ Step 4 complete.")

## 🖼️ Cell 10 — Step 5: SDXL Panel Image Generation (GPU)

This is the GPU-intensive step. Each panel takes ~30–90 seconds on a T4.

In [ ]:
import subprocess, sys, os

print("=" * 70)
print(f"STEP 5/5: Generating comic panels for Page {PAGE_TO_RENDER} with SDXL")
print("         Using CUDA GPU — T4 recommended")
print("=" * 70)

# Verify CUDA is available before launching
import torch
if not torch.cuda.is_available():
    print("⚠️  WARNING: CUDA is not available. Panel generation will be very slow on CPU.")
    print("   Please change runtime to GPU: Runtime → Change runtime type → T4 GPU")
else:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU detected: {gpu_name} ({vram_gb:.1f} GB VRAM)")

result = subprocess.run(
    [sys.executable, "generate_panels.py", "--page", str(PAGE_TO_RENDER)],
    cwd=os.path.join(os.getcwd(), "sdxl_code"),
    capture_output=False,
    text=True
)

if result.returncode != 0:
    raise RuntimeError("❌ SDXL panel generation failed! Check the output above.")

print("\n✅ Step 5 complete — all panels generated!")

## 🖼️ Cell 11 — Visualise the Comic Page

In [ ]:
import os, glob
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

comics_dir = os.path.join(os.getcwd(), "outputs", "comics")

# ── Load individual panels ─────────────────────────────────────────────────────
panel_pattern = os.path.join(comics_dir, f"page_{PAGE_TO_RENDER}_panel_sdxl_base_*.png")
panel_files   = sorted(glob.glob(panel_pattern))

# ── Load horizontal strip ──────────────────────────────────────────────────────
strip_path = os.path.join(comics_dir, f"page_{PAGE_TO_RENDER}_layout_sdxl_base_horizontal.png")
grid_path  = os.path.join(comics_dir, f"page_{PAGE_TO_RENDER}_layout_sdxl_base_grid.png")

# ── Display ────────────────────────────────────────────────────────────────────
if not panel_files:
    print(f"⚠️  No panels found for page {PAGE_TO_RENDER}. Make sure Step 5 completed.")
else:
    n = len(panel_files)
    print(f"✅ Found {n} panel(s) for page {PAGE_TO_RENDER}")

    # ── Individual Panels ──────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 6))
    if n == 1:
        axes = [axes]
    for ax, pf in zip(axes, panel_files):
        img = Image.open(pf)
        ax.imshow(img)
        ax.set_title(os.path.basename(pf).replace(".png", ""), fontsize=9)
        ax.axis("off")
    fig.suptitle(f"🎨 {CHARACTER_NAME} × {STORY_WORLD} — Page {PAGE_TO_RENDER} Panels",
                 fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

    # ── Horizontal Strip ───────────────────────────────────────────────────────
    if os.path.exists(strip_path):
        strip = Image.open(strip_path)
        fig, ax = plt.subplots(figsize=(18, 5))
        ax.imshow(strip)
        ax.set_title(f"Horizontal Strip — Page {PAGE_TO_RENDER}", fontsize=12)
        ax.axis("off")
        plt.tight_layout()
        plt.show()

    # ── Grid Layout ────────────────────────────────────────────────────────────
    if os.path.exists(grid_path):
        grid = Image.open(grid_path)
        fig, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(grid)
        ax.set_title(f"Grid Layout — Page {PAGE_TO_RENDER}", fontsize=12)
        ax.axis("off")
        plt.tight_layout()
        plt.show()

    print("\n✅ Visualisation complete!")

## 📊 Cell 12 — Storyboard Text Preview (All 10 Pages)

In [ ]:
import json, os

fusion_file = os.path.join(os.getcwd(), "outputs", "fusion", "fusion_complete.json")

with open(fusion_file) as f:
    fusion_data = json.load(f)

fusion = fusion_data.get("fusion", {})
pages  = fusion.get("storyboard_10_pages", [])

print("=" * 70)
print(f"STORYBOARD: {CHARACTER_NAME} × {STORY_WORLD}")
print(f"Visual Look: {fusion.get('character_visual_looks', '')[:150]}")
print("=" * 70)

for page in pages:
    pn   = page.get("page_number")
    loc  = page.get("location", "")
    narr = page.get("narrative_progression", "")
    mood = page.get("personality_state", "")
    panels = page.get("panels_breakdown", [])
    dialogues = page.get("dialogue_and_captions", [])

    print(f"\n{'─'*60}")
    print(f"📖 PAGE {pn}: {loc}")
    print(f"   Narrative : {narr[:120]}")
    print(f"   Mood State: {mood[:100]}")
    print(f"   Panels    :")
    for p in panels:
        print(f"     • {p[:100]}")
    if dialogues:
        print(f"   Dialogue  :")
        for d in dialogues[:3]:
            print(f"     💬 {d[:100]}")

print("\n" + "=" * 70)

## 😮 Cell 13 — Emotion Map Preview (All Pages)

In [ ]:
import json, os

emotions_file = os.path.join(os.getcwd(), "outputs", "fusion", "storyboard_with_emotions.json")

with open(emotions_file) as f:
    em_data = json.load(f)

pages = em_data.get("storyboard_with_emotions", [])
char_name = em_data.get("personality", {}).get("character_name", "Character")

print("=" * 70)
print(f"EMOTION MAP: {char_name} across 10 pages")
print("=" * 70)

for page in pages:
    pn = page.get("page_number")
    print(f"\n📖 Page {pn}: {page.get('location', '')}")
    for pd in page.get("panels_detail", []):
        pnum = pd.get("panel_number")
        emos = pd.get("emotions", {})
        action = pd.get("core_action", "")[:80]
        for char, emo in emos.items():
            if isinstance(emo, dict):
                emotion    = emo.get("emotion", "neutral")
                intensity  = emo.get("intensity", "medium")
                expr       = emo.get("expression_trigger", "")[:80]
                # Emoji map
                emoji = {"angry":"😡","furious":"🤬","fearful":"😨","terrified":"😱",
                         "sad":"😢","joyful":"😄","ecstatic":"🤩","surprised":"😲",
                         "curious":"🤔","neutral":"😐","determined":"😤"}.get(emotion, "🎭")
                print(f"   Panel {pnum} | {char}: {emoji} {emotion} [{intensity}]")
                print(f"           Expr: {expr}")

print("\n" + "=" * 70)

## 💾 Cell 14 — Download All Outputs as ZIP

In [ ]:
import os, zipfile
from google.colab import files

OUTPUT_DIR  = os.path.join(os.getcwd(), "outputs")
ZIP_PATH    = "/content/indie_comic_outputs.zip"
CHAR_SLUG   = CHARACTER_NAME.replace(" ", "_").lower()
WORLD_SLUG  = STORY_WORLD.replace(" ", "_").lower()

print("📦 Packing outputs into ZIP...")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk(OUTPUT_DIR):
        for fname in fnames:
            fpath = os.path.join(root, fname)
            arcname = os.path.relpath(fpath, os.path.dirname(OUTPUT_DIR))
            zf.write(fpath, arcname)

size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f"✅ ZIP created: {ZIP_PATH} ({size_mb:.1f} MB)")
print("⬇️  Downloading to your machine...")
files.download(ZIP_PATH)

---

## 🔁 Cell 15 — (Optional) Generate a Different Page

After the pipeline has run, you can regenerate panels for any page (1–10) without re-running the LLM steps.

In [ ]:
# ── Change PAGE_NUMBER and run this cell to render a different page ────────────
PAGE_NUMBER = 2   # <── set any page from 1 to 10

import subprocess, sys, os

result = subprocess.run(
    [sys.executable, "generate_panels.py", "--page", str(PAGE_NUMBER)],
    cwd=os.path.join(os.getcwd(), "sdxl_code"),
    capture_output=False, text=True
)

if result.returncode != 0:
    raise RuntimeError(f"❌ Panel generation failed for page {PAGE_NUMBER}.")

# ── Quick preview ──────────────────────────────────────────────────────────────
import glob
from PIL import Image
import matplotlib.pyplot as plt

comics_dir = os.path.join(os.getcwd(), "outputs", "comics")
panel_files = sorted(glob.glob(os.path.join(comics_dir, f"page_{PAGE_NUMBER}_panel_sdxl_base_*.png")))

n = len(panel_files)
fig, axes = plt.subplots(1, n, figsize=(6 * n, 6))
if n == 1: axes = [axes]
for ax, pf in zip(axes, panel_files):
    ax.imshow(Image.open(pf))
    ax.set_title(os.path.basename(pf), fontsize=8)
    ax.axis("off")
plt.suptitle(f"Page {PAGE_NUMBER} Panels", fontsize=13)
plt.tight_layout()
plt.show()

---

## 🎨 Cell 16 — (Optional) Generate Component Asset Images

Renders the persistent visual components (character sheet, environment, props) identified by the fusion engine.

In [ ]:
import subprocess, sys, os, glob
from PIL import Image
import matplotlib.pyplot as plt

print("🎨 Generating persistent component assets...")

result = subprocess.run(
    [sys.executable, "run_sdxl_pipeline.py"],
    cwd=os.path.join(os.getcwd(), "sdxl_code"),
    capture_output=False, text=True
)

if result.returncode != 0:
    raise RuntimeError("❌ Component generation failed!")

comics_dir = os.path.join(os.getcwd(), "outputs", "comics")
char_dir   = os.path.join(os.getcwd(), "outputs", "characters")

# Show character reference
char_ref = os.path.join(char_dir, "character_reference.png")
if os.path.exists(char_ref):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(Image.open(char_ref))
    ax.set_title(f"Character Reference: {CHARACTER_NAME}", fontsize=12)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

# Show component sheet
sheet_candidates = sorted(glob.glob(os.path.join(comics_dir, "component_sheet_*.png")))
for sheet in sheet_candidates:
    fig, ax = plt.subplots(figsize=(16, 6))
    ax.imshow(Image.open(sheet))
    ax.set_title(os.path.basename(sheet), fontsize=10)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

print("✅ Component generation visualised!")